# Exploratory Data Analysis

Import packages.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd

from tqdm import tqdm
from utils.common import DATASETS, DATASET_TO_NUM_INSTANCES
from utils.model import load_eval_results
from utils.path import resolve_dataset_path, resolve_eval_results_dir

## Datasets

View the datasets we're interested in:
- [DS-1000](https://huggingface.co/datasets/xlangai/DS-1000)
- [MATH](https://huggingface.co/datasets/EleutherAI/hendrycks_math)
- [MMLU](https://huggingface.co/datasets/cais/mmlu) 

**Note:** we're not interested in  Chatbot-Arena/Chatbot-Arena_NEW and WildChat10K beacuse they evaluate preferences between only two models on each instance.

In [2]:
DATASETS

['DS-1000', 'MATH', 'MMLU']

In [ ]:
name = "MMLU"

current_df = pd.read_json(resolve_dataset_path(name))

,question,[gpt-4o-mini]_answer
0,Question: Find the degree for the given field ...,To find the degree of the field extension \( \...
1,"Question: Let p = (1, 2, 5, 4)(2, 3) in S_5 . ...",To find the index of the subgroup generated by...
2,Question: Find all zeros in the indicated fini...,To find the zeros of the polynomial \( x^5 + 3...
3,Question: Statement 1 | A factor group of a no...,To analyze the statements:\n\n1. **Statement 1...
4,Question: Find the product of the given polyno...,To find the product of the polynomials \( f(x)...


In [ ]:
from dataset import load_dataset

Load each dataset.

In [ ]:
dataset_to_df = {}

kwargs = {
    "desc": "Loading datasets",
    "total": len(DATASETS),
    "unit": "dataset",
}

for dataset in tqdm(DATASETS, **kwargs):
    file_path = resolve_dataset_path(dataset)
    dataset_to_df[dataset] = pd.read_json(file_path)

View some of the instances in each dataset and ensure each dataset has the correct number  of instances.

In [ ]:
for dataset, df in dataset_to_df.items():
    print(f"{dataset}: {len(df)} instances")
    display(df.head())

    num_instances = DATASET_TO_NUM_INSTANCES[dataset]
    assert (
        len(df) == num_instances
    ), f"{dataset} has {len(df)} instances, but {num_instances} were expected"

View some of the instances in eac

## Evaluation Results

For each dataset, combine all of its evaluation results into a single CSV file.

In [ ]:
dataset_to_eval_results = {}

kwargs = {
    "desc": "Loading evaluation results",
    "total": len(DATASETS),
    "unit": "dataset",
}

for dataset in tqdm(DATASETS, **kwargs):
    eval_results_dir = resolve_eval_results_dir(dataset)
    model_results = {}

    for model_dir in eval_results_dir.iterdir():
        if not model_dir.is_dir():
            continue
        results_file = model_dir / "results.json"
        with open(results_file) as f:
            model_results[model_dir.name] = json.load(f)

    df = pd.DataFrame(model_results)
    dataset_to_eval_results[dataset] = df
    df.to_csv(eval_results_dir / "all.csv", index=False)

View each dataset's evaluation results.

In [ ]:
for dataset, df in dataset_to_eval_results.items():
    print(f"{dataset}: {len(df)} instances")
    display(df.head())